# Data Creation Playground

Full multi-depth pruning pipeline. G = Gemma-4 (Colab GPU), D = Gemini Flash Lite (cloud).
Source = `avreymi/reasoning-spectrum-qa` (1000 diverse QA across 6 reasoning families); each
question is assembled with its context and choices via `format_spectrum_question`, and answer
fields are never shown to G.

**Before running:** Enable GPU runtime → Runtime → Change runtime type → T4 GPU (or A100).

In [ ]:
!git clone https://github.com/avrymi-asraf/reasoning-pruning.git
%cd reasoning-pruning
# Gemma 4 requires transformers from git main — not yet in a stable PyPI release
%pip install -q "git+https://github.com/huggingface/transformers.git" "accelerate>=0.34.0" "datasets>=4.8.5" "torchvision>=0.27.0" "pyyaml>=6.0.2"

: 

: 

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
import os
os.environ["HF_TOKEN"] = input("Enter your Hugging Face token: ")
os.environ["GEMINI_API_KEY"] = input("Enter your Gemini API key: ")

In [ ]:
import os
import sys
sys.path.insert(0, "src")

from pathlib import Path
from dataclasses import replace

from reasoning_pruning.data_creation import (
    load_data_creation_config,
    load_questions,
    build_pt_dataset,
    build_rows_for_question,
)
from reasoning_pruning.clients import (
    TransformersGenerator,
    create_decision_model_from_config,
    load_prompt_template,
)

print("Imports OK")

In [ ]:
# Downloads ~5GB from Hub — takes 1-2 min on first run
generator = TransformersGenerator(
    source_model="avreymi/gemma-4-E2B-it-reasoning-pruning",
    generation_config={"max_new_tokens": 100, "temperature": 0.7, "do_sample": True},
    max_units_per_batch=2,
)
print("G ready:", generator.source_model)

In [ ]:
# Load config from YAML, then override depth/limit for quick playground runs.
# Source is avreymi/reasoning-spectrum-qa — questions are pulled straight from it,
# with context + choices already assembled by format_spectrum_question.
config = load_data_creation_config(Path("configs/data/dataset_builder_spectrum_gemma4.yaml"))
config = replace(config, max_pruning_depth=4, max_examples_per_question=4, source_limit=8)

questions = load_questions(config, hf_token=os.environ.get("HF_TOKEN"))

print(f"Config: max_depth={config.max_pruning_depth}, G={config.generator['model_id']}")
print(f"Source: {config.source_dataset} (split={config.source_split})")
print(f"Questions: {len(questions)}")
print("\n--- Example assembled question (context + choices, no answer) ---\n")
print(questions[0])

## D-prompt workflow

Iterate the decision-model prompt without leaving the notebook — no extra dependencies. Three cells:

1. **List / view** — see every `prompts/*.txt` and print any one inline with `show_prompt("<stem>")`.
2. **Write / edit** — set `PROMPT_NAME` + `PROMPT_TEXT` to create a new version (or overwrite an existing one with `OVERWRITE = True`). Only `{prompt_version} {question} {context} {reasoning_units}` may appear as single braces; escape any others as `{{ }}`.
3. **Choose** — set `PROMPT_VERSION` to any stem; it overrides `config.decision["prompt_version"]` and rebuilds D from the config (the production path). Re-run the build cells below to compare removal rates across versions.

> **D model note:** D is now built from `config.decision`, so it uses the model in `configs/data/dataset_builder_spectrum_gemma4.yaml` — currently `gemini-3.1-flash-lite` (matches `CLAUDE.md`). This replaces the notebook's old hardcoded `gemini-flash-lite-latest`. If `gemini-3.1-flash-lite` is not valid for your key, change `model_id` in that YAML.

In [ ]:
# LIST / VIEW prompts. PROMPTS_DIR resolves against the kernel's working dir —
# the printed absolute path must point at the repo's prompts/ folder. If it does
# not (e.g. VSCode started the kernel in the notebook's own dir), the import and
# config cells above would already have failed; fix the kernel cwd to the repo root.
PROMPTS_DIR = "prompts"


def list_prompts() -> list[str]:
    return sorted(p.stem for p in Path(PROMPTS_DIR).glob("*.txt"))


def show_prompt(version: str) -> None:
    print(load_prompt_template(version, PROMPTS_DIR))


print("prompts/ resolves to:", Path(PROMPTS_DIR).resolve())
print("\nAvailable prompt versions:")
for name in list_prompts():
    print("  -", name)

# Read one inline before choosing/editing:
# show_prompt("conservative-skip-v2-general")

In [ ]:
# WRITE / EDIT a D prompt without leaving the notebook.
#   - New version : set PROMPT_NAME to a fresh stem (e.g. conservative-skip-v3), run.
#   - Edit existing: point PROMPT_NAME at it, edit PROMPT_TEXT, set OVERWRITE = True.
# OVERWRITE defaults to False so "Run All" never clobbers an existing prompt file.
#
# REQUIRED placeholders (filled by format_decision_prompt via str.format):
#   {prompt_version} {question} {context} {reasoning_units}
# These are the ONLY allowed single braces. Any other literal { or } you add
# (e.g. a JSON example) MUST be doubled as {{ }} or .format() raises KeyError.
PROMPT_NAME = "conservative-skip-v3"
OVERWRITE = False

PROMPT_TEXT = r"""Decision prompt version: {prompt_version}

You are a conservative pruning decision model. Your job: find the first reasoning unit that is pure filler and can be removed without any loss of correctness.

These reasoning traces span many domains — arithmetic, science, commonsense, factual recall, multi-hop synthesis, and reading comprehension — NOT only math. "Real reasoning" therefore includes any of: a numeric computation, a derived fact, a logical deduction, eliminating or selecting an answer choice with a stated reason, or recalling a specific concrete fact needed for the answer.

STRICT REMOVAL CONDITIONS — all must hold:
1. The unit is pure filler: it contains NO computation, NO derived fact, NO logical deduction, NO answer-choice reasoning, and NO concrete recalled fact. It only restates the problem, announces intent, gives meta-commentary, or is a numbering/bullet artifact.
2. The unit at index removed_end_index+1 — which becomes the training target — contains ACTUAL reasoning as defined above. It must NOT be another goal/intent statement ('Determine X', 'We need to find Y', "Let's look at the options", 'Now recall who Y is').
3. Removing the span leaves the reasoning coherent.

REMOVABLE examples: '1.' (bare numbering), 'Let me think.' (filler), 'This follows the standard approach.' (commentary), "Let's consider each option." (intent), 'I need to find the answer.' (restatement).
NOT REMOVABLE as a target (these are intent, not reasoning): 'Convert 50 minutes to hours.', 'Determine the rate per minute.', 'Now eliminate the wrong choices.'
VALID targets (real reasoning — keep them, point at them): '50/60 = 5/6 hours.', 'Option B fits because the context says they reached an agreement.', 'The hypothalamus is the gland named in the passage.', 'Parton re-recorded "I Will Always Love You", so that is the song.'

If the next unit after the candidate removal is itself a goal/intent statement, set has_removal=false.

Question:
{question}

Current context:
{context}

Reasoning units:
{reasoning_units}

Return only JSON: has_removal (bool), removed_start_index (int), removed_end_index (int), reason (string), can_continue_after_skip (bool).
Set can_continue_after_skip=true only when the unit at removed_end_index+1 contains actual reasoning (computation, deduction, answer-choice reasoning, or a concrete recalled fact) — never a goal or intent statement.
"""

path = Path(PROMPTS_DIR) / f"{PROMPT_NAME}.txt"
if path.exists() and not OVERWRITE:
    print(f"{path} already exists — set OVERWRITE = True to replace it.")
else:
    path.write_text(PROMPT_TEXT)
    print(f"Wrote {path} ({len(PROMPT_TEXT)} chars). Select it in the next cell with PROMPT_VERSION = {PROMPT_NAME!r}.")

In [ ]:
# CHOOSE the prompt D uses. Set PROMPT_VERSION to any stem from the list above.
# This overrides config.decision["prompt_version"] and rebuilds D from the config
# — the same path the CLI and HF Jobs use, so D runs at config.pruning's
# temperature: 0.0 for fair prompt-to-prompt comparison.
PROMPT_VERSION = "conservative-skip-v2-general"

config.decision["prompt_version"] = PROMPT_VERSION
decision_model = create_decision_model_from_config(
    config.decision, config.pruning, prompts_dir=PROMPTS_DIR
)

print("D ready:", config.decision["model_id"], "| prompt:", config.decision["prompt_version"])
print("-" * 70)
show_prompt(PROMPT_VERSION)

In [ ]:
def show_rows(rows: list[dict]) -> None:
    if not rows:
        print("No rows generated — D found no safe removals at any depth")
        return
    for row in rows:
        sep = "=" * 70
        print(f"\n{sep}")
        print(f"  Depth {row['pruning_depth']}")
        print(f"{sep}")
        print(f"\n[GENERATED UNITS]")
        for i, u in enumerate(row["generated_units"]):
            marker = "  ✗" if row["metadata"]["removed_start_index"] <= i <= row["metadata"]["removed_end_index"] else "   "
            print(f"{marker} {i}: {u}")
        print(f"\n[REMOVED] indices {row['metadata']['removed_start_index']}–{row['metadata']['removed_end_index']}")
        print(f"  Reason: {row['metadata']['decision_reason']}")
        print(f"\n[INPUT_X]")
        for line in row["input_x"].splitlines():
            print(f"  {line}")
        print(f"\n[TARGET_Y]")
        print(f"  {row['target_y']}")
    print(f"\n  → {len(rows)} training row(s) from this question")

In [ ]:
# Run on a single question to inspect each depth in detail
rows = build_rows_for_question(
    question=questions[0],
    generator=generator,
    decision_model=decision_model,
    config=config,
)
show_rows(rows)

In [ ]:
# Run all questions and get a summary
all_rows = build_pt_dataset(
    questions=questions,
    generator=generator,
    decision_model=decision_model,
    config=config,
)

print(f"Total rows: {len(all_rows)}")
print(f"Rows per question: {len(all_rows) / len(questions):.1f} avg")
print(f"Depths seen: {sorted(set(r['pruning_depth'] for r in all_rows))}")
print()
for row in all_rows:
    q_preview = row["question"][:60]
    removed = " | ".join(row["metadata"]["removed_span"])
    print(f"[d={row['pruning_depth']}] {q_preview}...")
    print(f"  removed: {removed[:80]}")
    print(f"  target : {row['target_y'][:80]}")